# Build conflict-pair dataset from UltraFeedback

Pipeline:
1. Load UltraFeedback (64k prompts × 4 responses each)
2. Pre-filter to high-divergence, decent-quality pairs
3. Use a local LLM via vLLM to infer contrasting instructions from each pair
4. Parse + filter for genuine conflicts
5. Export as JSONL ready for Exp 2

Run on Colab (A100) or RunPod with a GPU that fits the model.

In [ ]:
!pip install -q vllm datasets huggingface_hub

In [ ]:
# Config
MODEL_ID = "google/gemma-4-26b-a4b-it"  # 26B MoE, ~4B active
MAX_MODEL_LEN = 8192
TENSOR_PARALLEL = 1          # bump if multi-GPU
GPU_MEMORY_UTILIZATION = 0.90

# Pre-filter thresholds
MIN_SCORE = 7.0              # both picked responses must score >= this
MIN_LENGTH_RATIO = 3.0       # max_len / min_len among responses (diversity proxy)
MIN_PROMPT_LEN = 20          # skip trivially short prompts
MAX_RESPONSE_CHARS = 2000    # truncate long responses to fit context

# Output
OUTPUT_PATH = "conflict_pairs.jsonl"
BATCH_SIZE = 256             # vLLM handles large batches well

# Optional: HF token for gated models
import os, getpass
if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass.getpass("HF Token (blank to skip): ").strip() or ""

## 1. Load and pre-filter UltraFeedback

In [ ]:
from datasets import load_dataset
from collections import defaultdict

ds = load_dataset("openbmb/UltraFeedback", split="train")
print(f"Total rows: {len(ds)}")

In [ ]:
def prefilter(row):
    """Keep rows where responses are diverse and at least two are decent quality."""
    if len(row["instruction"].strip()) < MIN_PROMPT_LEN:
        return False
    comps = row["completions"]
    if len(comps) < 2:
        return False
    scores = [c["overall_score"] for c in comps]
    # at least 2 responses with decent scores
    if sum(1 for s in scores if s >= MIN_SCORE) < 2:
        return False
    # response length diversity
    lens = [len(c["response"]) for c in comps]
    if min(lens) < 10:
        return False
    if max(lens) / (min(lens) + 1) < MIN_LENGTH_RATIO:
        return False
    return True

filtered = [row for row in ds if prefilter(row)]
print(f"After pre-filter: {len(filtered)} / {len(ds)} prompts")

## 2. Build LLM prompts for instruction inference

In [ ]:
ORACLE_SYSTEM = """You analyze pairs of assistant responses to the same prompt and infer what contrasting instructions could have produced each one.

Rules:
- The two instructions must be in GENUINE CONFLICT — following one must preclude following the other.
- Instructions should describe HOW to respond (style, format, approach, method), not WHAT to say.
- Keep instructions concise (1-2 sentences each).
- If the responses only differ in quality/correctness (not approach), set genuinely_conflicting to false.
- If the responses only differ in length/verbosity with no structural difference, set genuinely_conflicting to false.

Return ONLY valid JSON, no other text:
{
  "instruction_a": "<instruction that would produce response A>",
  "instruction_b": "<instruction that would produce response B>",
  "conflict_axis": "<short label: e.g. format, approach, code_style, tone, specificity, structure>",
  "genuinely_conflicting": true or false,
  "why_conflicting": "<one sentence explaining the incompatibility>"
}"""


def pick_best_pair(completions):
    """From N completions, pick the 2 with highest score that differ most in length."""
    decent = [c for c in completions if c["overall_score"] >= MIN_SCORE]
    if len(decent) < 2:
        decent = sorted(completions, key=lambda c: -c["overall_score"])[:2]
    # pick pair with max length ratio
    best_pair = None
    best_ratio = 0
    for i in range(len(decent)):
        for j in range(i + 1, len(decent)):
            la, lb = len(decent[i]["response"]), len(decent[j]["response"])
            ratio = max(la, lb) / (min(la, lb) + 1)
            if ratio > best_ratio:
                best_ratio = ratio
                best_pair = (decent[i], decent[j])
    return best_pair


def truncate(text, max_chars=MAX_RESPONSE_CHARS):
    if len(text) <= max_chars:
        return text
    return text[:max_chars] + "...[truncated]"


def build_oracle_prompt(instruction, comp_a, comp_b):
    return (
        f"## User prompt\n{truncate(instruction, 1000)}\n\n"
        f"## Response A (model: {comp_a['model']})\n{truncate(comp_a['response'])}\n\n"
        f"## Response B (model: {comp_b['model']})\n{truncate(comp_b['response'])}\n\n"
        f"Analyze the two responses and return the JSON as specified."
    )


# Build all oracle prompts
oracle_inputs = []
for row in filtered:
    pair = pick_best_pair(row["completions"])
    if pair is None:
        continue
    comp_a, comp_b = pair
    oracle_inputs.append({
        "original_prompt": row["instruction"],
        "source": row["source"],
        "model_a": comp_a["model"],
        "model_b": comp_b["model"],
        "response_a": comp_a["response"],
        "response_b": comp_b["response"],
        "score_a": comp_a["overall_score"],
        "score_b": comp_b["overall_score"],
        "oracle_prompt": build_oracle_prompt(row["instruction"], comp_a, comp_b),
    })

print(f"Oracle prompts built: {len(oracle_inputs)}")
print(f"\n--- Example oracle prompt ---")
print(oracle_inputs[0]["oracle_prompt"][:1500])

## 3. Run inference with vLLM

In [ ]:
from vllm import LLM, SamplingParams

llm = LLM(
    model=MODEL_ID,
    max_model_len=MAX_MODEL_LEN,
    tensor_parallel_size=TENSOR_PARALLEL,
    gpu_memory_utilization=GPU_MEMORY_UTILIZATION,
    trust_remote_code=True,
)
tokenizer = llm.get_tokenizer()
print(f"Model loaded: {MODEL_ID}")

In [ ]:
sampling_params = SamplingParams(
    temperature=0.0,
    max_tokens=512,
    stop=["\n}\n"],        # stop right after closing brace
    include_stop_str_in_output=True,
)

# Format as chat messages for the model's chat template
def format_chat(oracle_prompt):
    return tokenizer.apply_chat_template(
        [
            {"role": "user", "content": ORACLE_SYSTEM + "\n\n" + oracle_prompt},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )


# Process in batches
all_outputs = []
for start in range(0, len(oracle_inputs), BATCH_SIZE):
    batch = oracle_inputs[start : start + BATCH_SIZE]
    prompts = [format_chat(item["oracle_prompt"]) for item in batch]
    results = llm.generate(prompts, sampling_params)
    for item, result in zip(batch, results):
        item["raw_output"] = result.outputs[0].text
        all_outputs.append(item)
    print(f"  batch {start // BATCH_SIZE + 1}: processed {len(all_outputs)} / {len(oracle_inputs)}")

print(f"\nDone. Total outputs: {len(all_outputs)}")

## 4. Parse outputs and filter for genuine conflicts

In [ ]:
import json
import re


def parse_json_output(raw: str) -> dict | None:
    """Extract JSON from model output, handling markdown fences and trailing text."""
    # Try to find a JSON block in markdown fences first
    m = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", raw, re.DOTALL)
    if m:
        raw = m.group(1)
    else:
        # Find first { to last }
        start = raw.find("{")
        end = raw.rfind("}")
        if start == -1 or end == -1 or end <= start:
            return None
        raw = raw[start : end + 1]
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        # Try fixing common issues: trailing commas, single quotes
        cleaned = re.sub(r",\s*}", "}", raw)
        cleaned = re.sub(r",\s*]", "]", cleaned)
        try:
            return json.loads(cleaned)
        except json.JSONDecodeError:
            return None


parsed = []
parse_failures = 0
not_conflicting = 0

for item in all_outputs:
    result = parse_json_output(item["raw_output"])
    if result is None:
        parse_failures += 1
        continue
    if not result.get("genuinely_conflicting", False):
        not_conflicting += 1
        continue
    if not result.get("instruction_a") or not result.get("instruction_b"):
        parse_failures += 1
        continue

    parsed.append({
        "original_prompt": item["original_prompt"],
        "source": item["source"],
        "s_instruction": result["instruction_a"],
        "u_instruction": result["instruction_b"],
        "conflict_axis": result.get("conflict_axis", "unknown"),
        "why_conflicting": result.get("why_conflicting", ""),
        "s_aligned_response": item["response_a"],
        "u_aligned_response": item["response_b"],
        "model_a": item["model_a"],
        "model_b": item["model_b"],
        "score_a": item["score_a"],
        "score_b": item["score_b"],
    })

print(f"Parse failures:     {parse_failures}")
print(f"Not conflicting:    {not_conflicting}")
print(f"Genuine conflicts:  {len(parsed)}")
print(f"Yield:              {len(parsed) / len(all_outputs) * 100:.1f}%")

In [ ]:
# Distribution of conflict axes
from collections import Counter

axis_counts = Counter(r["conflict_axis"] for r in parsed)
print("Conflict axis distribution:")
for axis, count in axis_counts.most_common(20):
    print(f"  {axis:30s} {count:5d}  ({count / len(parsed) * 100:.1f}%)")

In [ ]:
# Inspect a few examples
import random
random.seed(42)
for item in random.sample(parsed, min(5, len(parsed))):
    print(f"--- {item['conflict_axis']} ---")
    print(f"PROMPT:  {item['original_prompt'][:150]}")
    print(f"S_INSTR: {item['s_instruction']}")
    print(f"U_INSTR: {item['u_instruction']}")
    print(f"WHY:     {item['why_conflicting']}")
    print()

## 5. Export

In [ ]:
with open(OUTPUT_PATH, "w") as f:
    for row in parsed:
        f.write(json.dumps(row) + "\n")

print(f"Wrote {len(parsed)} conflict pairs to {OUTPUT_PATH}")
print(f"File size: {os.path.getsize(OUTPUT_PATH) / 1024 / 1024:.1f} MB")

In [ ]:
# Optional: copy to Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    import shutil
    dest = '/content/drive/MyDrive/mech_spoof_results/conflict_pairs.jsonl'
    shutil.copy(OUTPUT_PATH, dest)
    print(f"Copied to {dest}")
except Exception as e:
    print(f"Drive copy skipped: {e}")